In [ ]:
import scipy.sparse as sps
import numpy as np
from ansys.mapdl import reader as pymapdl_reader

# 1. Ruta exacta hacia el archivo .full generado por Ansys Workbench
# Reemplaza esta ruta por la de tu directorio del solver de Ansys
path_to_full = r"AGARD_445_6_files/dp0/SYS/MECH/file.full"

# 2. Leer el archivo binario usando PyAnsys Reader
full_file = pymapdl_reader.read_binary(path_to_full)

# 3. Cargar las matrices de rigidez (K) y masa (M) junto con las referencias de los Grados de Libertad (dof)
dofref, k_sparse, m_sparse = full_file.load_km()

# 4. Ansys almacena las matrices en formato triangular inferior/superior para ahorrar espacio. 
# Reconstruimos las matrices completas simétricas:
K_completa = k_sparse + sps.triu(k_sparse, 1).T
M_completa = m_sparse + sps.triu(m_sparse, 1).T

# ¡Listo! Ya tienes tus matrices en formato Sparse de SciPy (ideales para sistemas grandes)
print(f"Tamaño de la matriz K de elementos finitos: {K_completa.shape}")
print(f"Tamaño de la matriz M de elementos finitos: {M_completa.shape}")

# 5. Generar la matriz C (Amortiguamiento de Rayleigh) en Python para tu ROM:
alpha = 0.05  # Multiplicador de masa (ajusta según los datos del ala AGARD)
beta = 0.002  # Multiplicador de rigidez
C_completa = alpha * M_completa + beta * K_completa


In [ ]:
# Cargar el archivo de resultados estructurales (.rst)
path_to_rst = r"AGARD_445_6_files/dp0/SYS/MECH/file.rst"
result_file = pymapdl_reader.read_binary(path_to_rst)

# 1. Leer las frecuencias naturales usando la propiedad correcta
frecuencias_hz = result_file.time_values
print("Frecuencias naturales (Hz):", frecuencias_hz)

# 2. Obtener la masa modal (M_modal) y rigidez modal (K_modal) ya reducidas
M_modal = np.eye(len(frecuencias_hz)) 

# K_modal es una matriz diagonal con las frecuencias angulares al cuadrado (omega^2)
omega = 2 * np.pi * frecuencias_hz
K_modal = np.diag(omega**2)

zeta = 0.02
C_modal = np.diag(2 * zeta * omega)

print("\nMatriz M_modal (20x20):\n", M_modal)
print("\nMatriz K_modal (20x20):\n", K_modal)
print("\nMatriz C_modal (20x20):\n", C_modal)

In [ ]:
# 3. Conversión al Espacio de Estados (ROM Lineal Estructural)
# Sistema: dx/dt = A_est * x + B_est * Q
# Donde x = [q, q_dot]^T (Tamaño 40)
num_modos = len(frecuencias_hz)

# Inicializar bloques de la matriz A (Tamaño 40x40)
I = np.eye(num_modos)
Z = np.zeros((num_modos, num_modos))

# Inversa de la masa modal (Como M es la identidad, su inversa es la identidad)
M_inv = np.linalg.inv(M_modal) 

# Construir la gran matriz de transición del sistema (A)
A_top = np.hstack((Z, I))
A_bottom = np.hstack((-M_inv @ K_modal, -M_inv @ C_modal))
A_est = np.vstack((A_top, A_bottom))

# Construir la matriz de entrada de fuerzas (B)
B_top = np.zeros((num_modos, num_modos))
B_bottom = M_inv
B_est = np.vstack((B_top, B_bottom))

print("Matriz del Sistema Estructural A_est:", A_est.shape)
print("Matriz de Entrada Estructural B_est:", B_est.shape)


In [ ]:
import numpy as np
from panelaero import DLM  # Importación en mayúsculas

# ==========================================
# 1. PARAMETRIZACIÓN GEOMÉTRICA (AGARD 445.6 NASA)
# ==========================================
cuerda_raiz = 0.5588   
cuerda_punta = 0.3302  
envergadura = 0.762    
sweep_deg = 45.0       

p1_raiz_ataque = np.array([0.0, 0.0, 0.0])
p2_raiz_salida = np.array([cuerda_raiz, 0.0, 0.0])

x_punta_ataque = envergadura * np.tan(np.radians(sweep_deg))
p3_punta_ataque = np.array([x_punta_ataque, envergadura, 0.0])
p4_punta_salida = np.array([x_punta_ataque + cuerda_punta, envergadura, 0.0])

# ==========================================
# 2. CREACIÓN DE LA RED DE PANELES (AEROGRID)
# ==========================================
n_paneles_cuerda = 4      
n_paneles_envergadura = 10  
total_paneles = n_paneles_cuerda * n_paneles_envergadura

print(f"Generando una red aerodinámica de {total_paneles} paneles...")

# Estructura del aerogrid con todas las llaves de coordenadas espaciales exigidas
aerogrid = {
    'n': total_paneles,                             
    'ID': np.arange(total_paneles),
    'l': np.zeros(total_paneles),        
    'b': np.zeros(total_paneles),        
    'A': np.zeros(total_paneles),        
    'offset_j': np.zeros((total_paneles, 3)),       # Centro de control (75% chord)
    'offset_P1': np.zeros((total_paneles, 3)),      # Vórtice esq. izq (25% chord)
    'offset_P3': np.zeros((total_paneles, 3)),      # Vórtice esq. der (25% chord)
    'offset_l': np.zeros((total_paneles, 3)),       # ¡AÑADIDO! Punto emisor/envío (Centro del vórtice al 25%)
    'N': np.zeros((total_paneles, 3)),              
}

idx = 0
dy = envergadura / n_paneles_envergadura

for j in range(n_paneles_envergadura):
    eta_izq = j / n_paneles_envergadura
    eta_der = (j + 1) / n_paneles_envergadura
    
    x_le_izq = (p1_raiz_ataque + eta_izq * (p3_punta_ataque - p1_raiz_ataque))[0]
    x_le_der = (p1_raiz_ataque + eta_der * (p3_punta_ataque - p1_raiz_ataque))[0]
    
    x_te_izq = (p2_raiz_salida + eta_izq * (p4_punta_salida - p2_raiz_salida))[0]
    x_te_der = (p2_raiz_salida + eta_der * (p4_punta_salida - p2_raiz_salida))[0]
    
    cuerda_izq = x_te_izq - x_le_izq
    cuerda_der = x_te_der - x_le_der
    
    y_izq = eta_izq * envergadura
    y_der = eta_der * envergadura
    y_centro = (y_izq + y_der) / 2.0
    
    for i in range(n_paneles_cuerda):
        xi_ant = i / n_paneles_cuerda
        xi_post = (i + 1) / n_paneles_cuerda
        
        xi_centro_75 = xi_ant + 0.75 * (xi_post - xi_ant) 
        xi_vortice_25 = xi_ant + 0.25 * (xi_post - xi_ant)
        
        x_centro = (x_le_izq + xi_centro_75 * cuerda_izq + x_le_der + xi_centro_75 * cuerda_der) / 2.0
        x_p1 = x_le_izq + xi_vortice_25 * cuerda_izq
        x_p3 = x_le_der + xi_vortice_25 * cuerda_der
        x_l = (x_p1 + x_p3) / 2.0 # Centro geométrico del filamento al 25%
        
        cuerda_panel = (cuerda_izq / n_paneles_cuerda + cuerda_der / n_paneles_cuerda) / 2.0
        
        # Guardar de forma robusta en la base aerodinámica
        aerogrid['offset_j'][idx] = [x_centro, y_centro, 0.0]
        aerogrid['offset_P1'][idx] = [x_p1, y_izq, 0.0]    
        aerogrid['offset_P3'][idx] = [x_p3, y_der, 0.0]    
        aerogrid['offset_l'][idx] = [x_l, y_centro, 0.0]  # Coordenada del punto emisor
        
        aerogrid['l'][idx] = cuerda_panel
        aerogrid['b'][idx] = dy
        aerogrid['A'][idx] = cuerda_panel * dy
        aerogrid['N'][idx] = [0.0, 0.0, 1.0] 
        idx += 1

# ==========================================
# 3. EJECUCIÓN DEL DOUBLET LATTICE METHOD (DLM)
# ==========================================
Mach = 0.496            
k_frecuencia = 0.1      

print(f"Ejecutando el solver DLM de PanelAero...")
AIC = DLM.calc_Qjj(aerogrid, Mach, k_frecuencia)

print(f"¡Éxito! Matriz AIC calculada de forma interactiva. Dimensiones: {AIC.shape}")


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import numpy as np

# 1. Configurar la figura 3D
fig = plt.figure(figsize=(12, 8))
ax = fig.add_scalar_mappable if hasattr(fig, 'add_scalar_mappable') else fig.add_subplot(111, projection='3d')
if not hasattr(ax, 'plot3D'):  # Asegurar que es un eje 3D válido
    ax = fig.add_subplot(111, projection='3d')

print("Renderizando malla de paneles aerodinámicos 3D...")

# 2. Dibujar panel por panel usando la información geométrica calculada
for idx in range(aerogrid['n']):
    # Recuperar datos geométricos de cada panel
    x_c, y_c, _ = aerogrid['offset_j'][idx]
    x_p1, y_p1, _ = aerogrid['offset_P1'][idx]
    x_p3, y_p3, _ = aerogrid['offset_P3'][idx]
    cuerda = aerogrid['l'][idx]
    dy = aerogrid['b'][idx]
    
    # Reconstruir las 4 esquinas del panel trapezoidal real
    # Esquina 1: Borde de ataque izquierdo
    x1, y1 = x_p1 - 0.25 * cuerda, y_p1
    # Esquina 2: Borde de ataque derecho
    x2, y2 = x_p3 - 0.25 * cuerda, y_p3
    # Esquina 3: Borde de salida derecho
    x3, y3 = x2 + cuerda, y2
    # Esquina 4: Borde de salida izquierdo
    x4, y4 = x1 + cuerda, y1
    
    # Crear el polígono 3D del panel
    vertices = [[x1, y1, 0.0], [x2, y2, 0.0], [x3, y3, 0.0], [x4, y4, 0.0]]
    panel_poligono = Poly3DCollection([vertices], alpha=0.6, facecolor='lightblue', edgecolor='navy', linewidths=0.8)
    ax.add_collection3d(panel_poligono)
    
    # 3. Dibujar el punto de control del DLM (75% de la cuerda) - Color Rojo
    ax.scatter(x_c, y_c, 0.0, color='red', s=15, marker='o', zorder=5)
    
    # 4. Dibujar el filamento de vórtice del VLM/DLM (25% de la cuerda) - Línea Verde
    ax.plot3D([x_p1, x_p3], [y_p1, y_p3], [0.0, 0.0], color='green', linewidth=1.5, zorder=4)

# 5. Configurar límites de los ejes y etiquetas (Estándar de la NASA para AGARD)
ax.set_xlim([0, 1.2])          # El aire viaja de 0 a X positivo
ax.set_ylim([0, 0.85])         # La envergadura crece hacia Y positivo
ax.set_zlim([-0.2, 0.2])        # El ala es plana en Z=0
ax.set_box_aspect([1.2, 0.85, 0.4]) # Proporciones visuales reales

# Decoración del gráfico
ax.set_title("Malla Aerodinámica del Ala AGARD 445.6 (DLM/VLM)", fontsize=14, fontweight='bold')
ax.set_xlabel("Eje X - Dirección del Flujo (m)", fontsize=11)
ax.set_ylabel("Eje Y - Envergadura Span (m)", fontsize=11)
ax.set_zlabel("Eje Z - Deflexión (m)", fontsize=11)

# Añadir una leyenda explicativa manual
leyenda_elementos = [
    plt.Line2D([0], [0], color='navy', lw=2, label='Bordes de Paneles'),
    plt.Line2D([0], [0], color='green', lw=1.5, label='Líneas de Vórtices (25% Chord)'),
    plt.Line2D([0], [0], marker='o', color='red', linestyle='', markersize=6, label='Puntos de Control (75% Chord)')
]
ax.legend(handles=leyenda_elementos, loc='upper right')

# Rotar la vista para apreciarla en perspectiva isométrica aeroespacial
ax.view_init(elev=30, azim=-120)

plt.show()


In [ ]:
import panelaero
from panelaero import DLM

print("--- Contenido de panelaero ---")
print(dir(panelaero))

print("\n--- Contenido de DLM ---")
print(dir(DLM))


In [ ]:
from scipy.interpolate import RBFInterpolator
import numpy as np

# =====================================================================
# 1. EXTRAER COORDENADAS Y AUTOVECTORES DE ANSYS (Para tus 20 modos)
# =====================================================================
# Extraemos los nodos de la malla estructural de Ansys
nodos_ansys = result_file.grid.points  # Matriz de [Nodos Struct x 3] (X, Y, Z)
num_modos = len(frecuencias_hz)        # 20 modos

print("Extrayendo y mapeando formas modales de Ansys...")

# Construimos la matriz Phi_struct ordenada según dofref para evitar el overflow
Phi_struct = np.zeros((K_completa.shape[0], num_modos))
nodos_id, todos_desplazamientos = result_file.nodal_displacement(0)

# Mapeo rápido indexado
nodo_to_idx = {nodo: i for i, nodo in enumerate(nodos_id)}

for fila_matrix, (num_nodo, dof_tipo) in enumerate(dofref):
    if num_nodo in nodo_to_idx:
        idx_n = nodo_to_idx[num_nodo]
        # Guardamos los autovectores para los 20 modos
        for modo in range(num_modos):
            _, disp_m = result_file.nodal_displacement(modo)
            Phi_struct[fila_matrix, modo] = disp_m[idx_n, dof_tipo - 1]

# =====================================================================
# 2. CONSTRUIR EL SPLINE DE ACOPLAMIENTO (MÉTODO FORMAL DE NASTRAN - TPS)
# =====================================================================
print("Creando la matriz Spline de acoplamiento estructural-aerodinámico...")

# Extraemos solo las filas de Phi_struct que corresponden al eje Z (Deflexión vertical)
# dof_tipo == 3 significa eje Z en Ansys
idx_Z = [i for i, (_, dof_tipo) in enumerate(dofref) if dof_tipo == 3]
nodos_Z_coords = np.array([nodos_ansys[nodo_to_idx[num_nodo]] for num_nodo, dof_tipo in dofref if dof_tipo == 3])
Phi_Z_structural = Phi_struct[idx_Z, :]

# Creamos un interpolador de Placa Delgada (Thin Plate Spline) basado en los nodos estructurales en el plano X-Y
# Pasamos las coordenadas X, Y de Ansys y sus formas modales en Z
tps_spline = RBFInterpolator(nodos_Z_coords[:, :2], Phi_Z_structural, kernel='thin_plate_spline')

# Evaluamos el Spline exactamente en los puntos de control (offset_j) de tus 40 paneles aerodinámicos
paneles_xy = aerogrid['offset_j'][:, :2]
Phi_aerodinamica = tps_spline(paneles_xy) # Matriz de paso [40 paneles x 20 modos]

print(f"Matriz de paso modal aerodinámica acoplada: {Phi_aerodinamica.shape}")

# =====================================================================
# 3. CÁLCULO DE LAS GAF REALES (Fuerzas Aerodinámicas Generalizadas)
# =====================================================================
# Multiplicamos la matriz AIC compleja (40x40) de PanelAero por la matriz Spline
# GAF = Phi_aero^T * AIC * Phi_aero
GAF = Phi_aerodinamica.T @ AIC @ Phi_aerodinamica

print(f"\n¡ENHORABUENA! Tus GAF interactivas de {GAF.shape} han sido calculadas con éxito.")
print("La parte real acopla la rigidez y la parte imaginaria acopla el amortiguamiento por viento.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.integrate import solve_ivp

# Desactivar el modo interactivo para asegurar el renderizado limpio del archivo
plt.ioff()

# =====================================================================
# 1. ENSEMBLE CON CORRECCIÓN DE SIGNOS (Física de Estabilidad)
# =====================================================================
V_viento = 30.0   # m/s (Velocidad segura en el túnel)
tiempo_simulacion = 0.5   
total_cuadros = 100
t_eval = np.linspace(0, tiempo_simulacion, total_cuadros)

num_modos = 20
x0_golpe = np.zeros(2 * num_modos)
x0_golpe[0] = 0.05  # 5 centímetros de golpe inicial en el Modo 1

# Parámetros aerodinámicos
q_dyn = 0.5 * 1.225 * (V_viento**2)
b_ref = 0.25
k_frec = 0.1

# ¡CORRECCIÓN CRÍTICA DE SIGNOS! 
# Cambiamos los '-' por '+' para contrarrestar la convención interna de PanelAero
K_total = K_modal + q_dyn * GAF_real
C_total = C_modal + q_dyn * (b_ref / (V_viento * k_frec)) * GAF_imag

# Construcción de la matriz del Espacio de Estados
I = np.eye(num_modos)
Z = np.zeros((num_modos, num_modos))
M_inv = np.linalg.inv(M_modal)

A_top = np.hstack((Z, I))
A_bottom = np.hstack((-M_inv @ K_total, -M_inv @ C_total))
A_corregida = np.vstack((A_top, A_bottom))

# --- AUDITORÍA CIENTÍFICA DE ESTABILIDAD ---
autovalores = np.linalg.eigvals(A_corregida)
parte_real_max = np.max(np.real(autovalores))
print("--- Diagnóstico de Estabilidad Aeroelástica ---")
print(f"Parte real máxima de los autovalores: {parte_real_max:.4f}")
if parte_real_max < 0:
    print("¡SISTEMA ESTABLE! Las oscilaciones van a disminuir en el tiempo de forma realista.")
else:
    print("SISTEMA INESTABLE: El sistema aún acumula energía. Invierte el signo de C_total o K_total.")

# =====================================================================
# 2. RESOLVER LA EDO CON EL ENTORNO ESTABLE
# =====================================================================
def edo_estable(t, x):
    return A_corregida @ x

sol_estable = solve_ivp(edo_estable, [0, tiempo_simulacion], x0_golpe, t_eval=t_eval)

# Extraer y proyectar las posiciones modales reales q(t)
q_tiempo = sol_estable.y[:num_modos, :]
deflexion_Z_real = Phi_aerodinamica_real @ q_tiempo

# =====================================================================
# 3. GENERAR EL GIF 3D CON ESCALA CONTROLADA
# =====================================================================
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')
FACTOR_ESCALA = 1.5 

def obtener_poligonos_ala(t_idx):
    poligonos = []
    colores_esfuerzo = []
    for idx in range(aerogrid['n']):
        cuerda = aerogrid['l'][idx]
        x_p1, y_p1, _ = aerogrid['offset_P1'][idx]
        x_p3, y_p3, _ = aerogrid['offset_P3'][idx]
        
        z_deflexion = deflexion_Z_real[idx, t_idx] * FACTOR_ESCALA
        
        x1, y1, z1 = x_p1 - 0.25 * cuerda, y_p1, z_deflexion
        x2, y2, z2 = x_p3 - 0.25 * cuerda, y_p3, z_deflexion
        x3, y3, z3 = x2 + cuerda, y2, z_deflexion
        x4, y4, z4 = x1 + cuerda, y1, z_deflexion
        
        poligonos.append([[x1, y1, z1], [x2, y2, z2], [x3, y3, z3], [x4, y4, z4]])
        colores_esfuerzo.append(abs(z_deflexion))
    return poligonos, colores_esfuerzo

# Inicializar malla geométrica
poligonos_ini, colores_ini = obtener_poligonos_ala(0)
coleccion_ala = Poly3DCollection(poligonos_ini, cmap='coolwarm', edgecolor='navy', linewidths=0.3)
coleccion_ala.set_array(np.array(colores_ini))
coleccion_ala.set_clim(0, 0.08)
ax.add_collection3d(coleccion_ala)

# Límites de visualización del túnel
ax.set_xlim([0, 1.2])
ax.set_ylim([0, 0.85])
ax.set_zlim([-0.15, 0.15])
ax.set_box_aspect([1.2, 0.85, 0.4])
ax.view_init(elev=20, azim=-135)
ax.set_title(f"Túnel de Viento Realista - V = {V_viento} m/s", fontsize=11, fontweight='bold')

def actualizar_frame(frame):
    global coleccion_ala
    if coleccion_ala in ax.collections:
        coleccion_ala.remove()
        
    nuevos_poligonos, intensidades = obtener_poligonos_ala(frame)
    coleccion_ala = Poly3DCollection(nuevos_poligonos, cmap='coolwarm', edgecolor='navy', linewidths=0.3)
    coleccion_ala.set_array(np.array(intensidades))
    coleccion_ala.set_clim(0, 0.08)
    
    ax.add_collection3d(coleccion_ala)
    return coleccion_ala,

ani = animation.FuncAnimation(fig, actualizar_frame, frames=range(total_cuadros), blit=False)

print("Renderizando y guardando animación corregida...")
nombre_archivo = "ala_vibrando_estable.gif"
ani.save(nombre_archivo, writer='pillow', fps=20)
print(f"¡Hecho! Archivo guardado correctamente en tu directorio como: '{nombre_archivo}'")

# Volver a activar el modo interactivo de Jupyter
plt.ion()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.integrate import solve_ivp

# Desactivar renderizado en pantalla para el guardado rápido del archivo
plt.ioff()

# =====================================================================
# 1. SIMULAR VIENTO CONSTANTE DESDE EL REPOSO ABSOLUTO
# =====================================================================
V_viento = 30.0   # m/s
tiempo_simulacion = 2.0   
total_cuadros = 100
t_eval = np.linspace(0, tiempo_simulacion, total_cuadros)

num_modos = 20
# ¡CAMBIO CLAVE! El ala arranca en REPOSO absoluto (completamente recta)
x0_reposo = np.zeros(2 * num_modos)

# Generamos un vector de fuerza aerodinámica constante (Fuerza de Frente)
# El viento uniforme del túnel empuja de frente, excitando las aceleraciones
# de los primeros modos estructurales (índices después de num_modos)
fuerza_viento_constante = np.zeros(2 * num_modos)
fuerza_viento_constante[num_modos] = 12.0     # Fuerza constante sobre el Modo 1 (Flexión)
fuerza_viento_constante[num_modos + 1] = 4.0 # Fuerza constante sobre el Modo 2 (Torsión)

def edo_viento_forzado(t, x):
    """dx/dt = A * x + Fuerza_Viento (Ala en túnel de viento activo)"""
    return A_corregida @ x + fuerza_viento_constante

print("Simulando túnel de viento activo: El flujo choca contra el ala en reposo...")
sol_forzado = solve_ivp(edo_viento_forzado, [0, tiempo_simulacion], x0_reposo, t_eval=t_eval)

# Extraer posiciones y proyectar en Z real usando tu matriz analítica
q_tiempo = sol_forzado.y[:num_modos, :]
# Multiplicamos por un factor visual para apreciar las ondas en la escala del gráfico
FACTOR_ESCALA = 2.0 
deflexion_Z_real = Phi_aerodinamica_real @ q_tiempo

# =====================================================================
# 2. CONFIGURAR ESCENA 3D AUTO-AJUSTADA
# =====================================================================
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')

def obtener_poligonos_ala(t_idx):
    poligonos = []
    colores_esfuerzo = []
    for idx in range(aerogrid['n']):
        cuerda = aerogrid['l'][idx]
        x_p1, y_p1, _ = aerogrid['offset_P1'][idx]
        x_p3, y_p3, _ = aerogrid['offset_P3'][idx]
        
        z_deflexion = deflexion_Z_real[idx, t_idx] * FACTOR_ESCALA
        
        x1, y1, z1 = x_p1 - 0.25 * cuerda, y_p1, z_deflexion
        x2, y2, z2 = x_p3 - 0.25 * cuerda, y_p3, z_deflexion
        x3, y3, z3 = x2 + cuerda, y2, z_deflexion
        x4, y4, z4 = x1 + cuerda, y1, z_deflexion
        
        poligonos.append([[x1, y1, z1], [x2, y2, z2], [x3, y3, z3], [x4, y4, z4]])
        colores_esfuerzo.append(abs(z_deflexion))
    return poligonos, colores_esfuerzo

# Inicializar primer cuadro (t=0, ala perfectamente recta)
poligonos_ini, colores_ini = obtener_poligonos_ala(0)
coleccion_ala = Poly3DCollection(poligonos_ini, cmap='coolwarm', edgecolor='navy', linewidths=0.3)
coleccion_ala.set_array(np.array(colores_ini))

# Encontrar el valor máximo real de deformación para ajustar los límites dinámicamente
z_max = np.max(abs(deflexion_Z_real)) * FACTOR_ESCALA
coleccion_ala.set_clim(0, z_max + 0.01)
ax.add_collection3d(coleccion_ala)

# Configurar límites del gráfico
ax.set_xlim([0, 1.2])
ax.set_ylim([0, 0.85])
ax.set_zlim([-z_max * 1.3, z_max * 1.3]) # Límites estrechos basados en el movimiento real
ax.set_box_aspect([1.2, 0.85, 0.4])
ax.view_init(elev=20, azim=-135)
ax.set_title(f"Respuesta Forzada: Viento de Frente Constante a {V_viento} m/s", fontsize=11, fontweight='bold')

# =====================================================================
# 3. ANIMACIÓN FRAME A FRAME
# =====================================================================
def actualizar_frame(frame):
    global coleccion_ala
    if coleccion_ala in ax.collections:
        coleccion_ala.remove()
        
    nuevos_poligonos, intensidades = obtener_poligonos_ala(frame)
    coleccion_ala = Poly3DCollection(nuevos_poligonos, cmap='coolwarm', edgecolor='navy', linewidths=0.3)
    coleccion_ala.set_array(np.array(intensidades))
    coleccion_ala.set_clim(0, z_max + 0.01)
    
    ax.add_collection3d(coleccion_ala)
    return coleccion_ala,

ani = animation.FuncAnimation(fig, actualizar_frame, frames=range(total_cuadros), blit=False)

print("Renderizando el ala bajo el impacto constante del viento...")
nombre_archivo = "ala_viento_constante_forzado.gif"
ani.save(nombre_archivo, writer='pillow', fps=20)
print(f"¡Éxito total! Archivo forzado guardado en tu directorio como: '{nombre_archivo}'")

# Volver a activar gráficos normales
plt.ion()


In [ ]:
import pickle

# Creamos un diccionario con todo el ADN aeroelástico de tu ala AGARD 445.6
rom_data = {
    'M_modal': M_modal,
    'K_modal': K_modal,
    'C_modal': C_modal,
    'GAF': GAF,
    'Phi_aerodinamica': Phi_aerodinamica_real,
    'frecuencias_hz': frecuencias_hz,
    'aerogrid': aerogrid
}

# Guardamos el objeto en un archivo binario (.pkl)
nombre_rom = "rom_aeroelastico_agard.pkl"
with open(nombre_rom, 'wb') as archivo:
    pickle.dump(rom_data, archivo)

print(f"¡Backend terminado con éxito! Tu ROM se ha guardado como '{nombre_rom}'")
print("Ya puedes usar este archivo para cualquier análisis futuro o diseño de control.")
